In [3]:
from pathlib import Path
import xarray as xr
import geopandas as gpd
import pandas as pd
from pandas import IndexSlice
import numpy as np
import re
import datetime
from typing import List
import os

In [4]:
# Constantes
IPATH = Path('./inputs_refuel/')
OPATH = Path('./outputs_refuel/')

# IPATH = Path('./inputs_reabastecimento_marcos/')
# OPATH = Path('./outputs_reabastecimento_marcos/')


In [5]:
ref_mesh = xr.open_dataset(IPATH / 'ref_mesh.nc')
ref_mesh

<xarray.Dataset> Size: 92MB
Dimensions:     (Time: 1, south_north: 420, west_east: 393,
                 south_north_stag: 421, west_east_stag: 394, land_cat: 21,
                 soil_cat: 16, month: 12, dust_erosion_dimension: 3)
Dimensions without coordinates: Time, south_north, west_east, south_north_stag,
                                west_east_stag, land_cat, soil_cat, month,
                                dust_erosion_dimension
Data variables: (12/56)
    Times       (Time) |S19 19B ...
    XLAT_M      (Time, south_north, west_east) float32 660kB ...
    XLONG_M     (Time, south_north, west_east) float32 660kB ...
    XLAT_V      (Time, south_north_stag, west_east) float32 662kB ...
    XLONG_V     (Time, south_north_stag, west_east) float32 662kB ...
    XLAT_U      (Time, south_north, west_east_stag) float32 662kB ...
    ...          ...
    OL4         (Time, south_north, west_east) float32 660kB ...
    VAR_SSO     (Time, south_north, west_east) float32 660kB ...
    LAKE_DEPTH  (Time, south_north, west_east) float32 660kB ...
    EROD        (Time, dust_erosion_dimension, south_north, west_east) float32 2MB ...
    CLAYFRAC    (Time, south_north, west_east) float32 660kB ...
    SANDFRAC    (Time, south_north, west_east) float32 660kB ...
Attributes: (12/51)
    TITLE:                           OUTPUT FROM GEOGRID V4.1
    SIMULATION_START_DATE:           0000-00-00_00:00:00
    WEST-EAST_GRID_DIMENSION:        394
    SOUTH-NORTH_GRID_DIMENSION:      421
    BOTTOM-TOP_GRID_DIMENSION:       0
    WEST-EAST_PATCH_START_UNSTAG:    1
    ...                              ...
    FLAG_LAI12M:                     1
    FLAG_VAR_SSO:                    1
    FLAG_LAKE_DEPTH:                 1
    FLAG_EROD:                       1
    FLAG_CLAYFRAC:                   1
    FLAG_SANDFRAC:                   1

In [6]:
# Assimilando as coordenadas do centróide
# xlat_c = ref_mesh.XLAT_C.sel(Time=0).values
# xlong_c = ref_mesh.XLONG_C.sel(Time=0).values

# Selecionando e cortando para os dados do MCIP
centroid_lat = ref_mesh.XLAT_M.sel(Time=0).values[1:-1,1:-1]
centroid_long = ref_mesh.XLONG_M.sel(Time=0).values[1:-1,1:-1]
# centroid_lat = ref_mesh.XLAT_M.sel(Time=0).values
# centroid_long = ref_mesh.XLONG_M.sel(Time=0).values

# Assimilando tags
south_north_stag = ref_mesh.south_north_stag.values
west_east_stag = ref_mesh.west_east_stag.values

In [7]:
ny, nx = ref_mesh.XLAT_C.sel(Time=0).shape
ny, nx = ny - 2, nx - 2

In [24]:
# Método iguais ao processo 9.1
def format_to_cmaq(df):
    """
    Add required dimensions to generate CMAQ input.
    
    Must be columns time and cell_id. Must be DAYLY DATA."""
    df = df.copy()

    # Criando coluna TSTEP
    df['TSTEP'] = df.time.dt.hour

    return df

def generate_final_dataframe(day_df):

    # Provisoriamente adicionando tstep (foi a forma mais fácil que consegui fazer)
    day_df = format_to_cmaq(day_df)

    cell_dimension = ['cell_id']
    time_dimensions = ['TSTEP']

    group_df = day_df.drop(columns='time')

    df = group_df.groupby(by=cell_dimension + time_dimensions).sum()

    time_values = []
    for time_dim in time_dimensions:
        time_values.append(
            df.index.get_level_values(time_dim).unique().sort_values())
        
    if len(time_values) != 1:
        time_index = pd.MultiIndex.from_arrays(
            time_values,
            names=tuple(time_dimensions)
        )
    else:
        time_index = time_values[0]

    # Criando daterange
    # date_list = [int(i) for i in re.findall('\\d+', (file.stem))]
    # start_date = datetime.datetime(
    #         dates_list[0], dates_list[1], dates_list[2], dates_list[3])
    # end_date = datetime.datetime(
    #         dates_list[4], dates_list[5], dates_list[6], dates_list[7])

    
    # kwargs = {'nx': nx, 'ny': ny, 'centroid_lat': centroid_lat}
    
    # Se é necessário exportar por dia
    # days = [i + 1 for i in range(pd.Period(str(date)).days_in_month)]
    # for day in days:
    # start_date = datetime.datetime(
    #     date_list[0], date_list[1], day, 0)
    # end_date = datetime.datetime(
    #     date_list[0], date_list[1], day, 23)
    
    # date_range = pd.date_range(
    #     start_date,
    #     end_date,
    #     freq='1h'
    # )

    # Criando dataframe com todos os ids de célula e datas
    full_mesh_df = pd.DataFrame(
        index=pd.MultiIndex.from_product(
            (range(0, ny*nx), time_index),
            names=tuple(cell_dimension + time_dimensions)
        )
    )

    # Unindo as informações
    full_mesh_df = full_mesh_df.merge(
        df,
        left_index=True,
        right_index=True,
        how='left'
    )

    return full_mesh_df
    

def generate_and_format_xarray_dataset(
        full_mesh_df: pd.DataFrame
    ):

    full_mesh_ds = full_mesh_df.to_xarray()

    full_mesh_ds = full_mesh_ds.assign_coords(
        y=("cell_id", np.repeat(np.arange(1, ny+1)[::-1], nx)),
        x=("cell_id", np.tile(np.arange(1, nx+1), ny))) \
            .set_index(cell_id=("y", "x")) \
                .unstack("cell_id")

    full_mesh_ds = full_mesh_ds \
        .isel(y=slice(0, -1), x=slice(0, -1)) \
        .assign_coords(
            lat=(('y', 'x'), centroid_lat),
            lon=(('y', 'x'), centroid_long)
        )
    
    # full_mesh_ds = full_mesh_ds.isel(y=slice(1, -1), x=slice(1, -1))

    return full_mesh_ds


def add_25th_hour(df_to_add, df_with_the_hour):

    df_to_add = df_to_add.reset_index(drop=False)
    df_with_the_hour = df_with_the_hour.reset_index(drop=False)
    
    first_hour_actual_day = df_with_the_hour.loc[
        df_with_the_hour['TSTEP'] == 0].copy()

    first_hour_actual_day.loc[:, 'TSTEP'] = 24

    last_day_full_df_mesh = pd.concat([
        df_to_add,
        first_hour_actual_day
    ], ignore_index=True)

    last_day_full_df_mesh = last_day_full_df_mesh.set_index([
        'cell_id', 'TSTEP'
    ])

    return last_day_full_df_mesh

def save_netcdf(opath, prefix, filename, dataset, compression=True):
    (opath / prefix).mkdir(
        parents=True, exist_ok=True)
    
    encoding=None
    
    # Force compressing
    if compression:
        comp = dict(zlib=True, complevel=5)
        encoding = {var: comp for var in dataset.data_vars}


    dataset.to_netcdf(
        opath /
        prefix /
        f'{filename}.nc',
        engine='netcdf4', # Before: 'scipy'
        encoding=encoding)


In [16]:
import os

In [25]:
# Coletando os arquivos de malha
# Storing last day to append hipothetical 25'th hour
last_day_full_df_mesh = None
prefix='finals'

tablePath = Path(os.path.join(os.path.dirname(os.path.dirname(os.getcwd())),'outputs','emissoes_brasil','emissoes_2023'))
sorted_files = sorted(tablePath.rglob('*.parquet'))

last_date = None

for id, file in enumerate(sorted_files):
    df = pd.read_parquet(file)
    df = df.reset_index(drop=False)
    try:
        df = df.sort_values(by=['cell_id', 'time'])
    except:
        df.loc[:, 'time'] = df['datetime']
        df = df.drop(columns='datetime')

    name_splitted = file.stem.split('-')
    year = int(name_splitted[0])
    month = int(name_splitted[1])
    day = int(name_splitted[2])

    actual_date = datetime.datetime(
        year,
        month,
        day
    )

    full_mesh_df = generate_final_dataframe(df)


    if last_day_full_df_mesh is None:
        print(f'Step 1: Adding {actual_date} as last date')
        last_day_full_df_mesh = full_mesh_df
        last_date = actual_date
        continue
    elif id < (len(sorted_files) - 1):
        print(f'Step 2: Processing {last_date}')
        # Setting first actual dataframe hour to 25 last dataframe
        # hour
        last_day_full_df_mesh = add_25th_hour(
            df_to_add=last_day_full_df_mesh,
            df_with_the_hour=full_mesh_df
        )

        full_mesh_ds = generate_and_format_xarray_dataset(
            last_day_full_df_mesh)
        
        last_day_full_df_mesh = full_mesh_df

        if last_date is None:
            print('Something is wrong! '
                  'Assuming that last file is from one day before.')
            last_date = actual_date - datetime.timedelta(days=1)
        
        save_netcdf(
            OPATH,
            prefix,
            f'{last_date.year}-{last_date.month:02d}-{last_date.day:02d}',
            full_mesh_ds
        )
        
    else:
        print(f'Step 3: Processing {last_date} and {actual_date}')
        # If last date:
        # 1. Set the last day 25th hour as 1th hour os actual
        # 2. Repeat the actual 1th hour to 25th hour of last record
        last_day_full_df_mesh = add_25th_hour(
            df_to_add=last_day_full_df_mesh,
            df_with_the_hour=full_mesh_df
        )

        full_mesh_ds = generate_and_format_xarray_dataset(
            last_day_full_df_mesh)
        
        last_date = actual_date - datetime.timedelta(days=1)
        
        save_netcdf(
            OPATH,
            prefix,
            f'{last_date.year}-{last_date.month:02d}-{last_date.day:02d}',
            full_mesh_ds
        )

        # Get first date as 25th TSTEP
        full_mesh_df = add_25th_hour(
            df_to_add=full_mesh_df,
            df_with_the_hour=full_mesh_df
        )

        full_mesh_ds = generate_and_format_xarray_dataset(
            last_day_full_df_mesh)
        
        save_netcdf(
            OPATH,
            prefix,
            f'{year}-{month:02d}-{day:02d}',
            full_mesh_ds
        )

    last_date = actual_date

Step 1: Adding 2023-01-01 00:00:00 as last date
Step 2: Processing 2023-01-01 00:00:00
Step 2: Processing 2023-01-02 00:00:00
Step 2: Processing 2023-01-03 00:00:00
Step 2: Processing 2023-01-04 00:00:00
Step 2: Processing 2023-01-05 00:00:00
Step 2: Processing 2023-01-06 00:00:00
Step 2: Processing 2023-01-07 00:00:00
Step 2: Processing 2023-01-08 00:00:00
Step 2: Processing 2023-01-09 00:00:00
Step 2: Processing 2023-01-10 00:00:00
Step 2: Processing 2023-01-11 00:00:00
Step 2: Processing 2023-01-12 00:00:00
Step 2: Processing 2023-01-13 00:00:00
Step 2: Processing 2023-01-14 00:00:00
Step 2: Processing 2023-01-15 00:00:00
Step 2: Processing 2023-01-16 00:00:00
Step 2: Processing 2023-01-17 00:00:00
Step 2: Processing 2023-01-18 00:00:00
Step 2: Processing 2023-01-19 00:00:00
Step 2: Processing 2023-01-20 00:00:00
Step 2: Processing 2023-01-21 00:00:00
Step 2: Processing 2023-01-22 00:00:00
Step 2: Processing 2023-01-23 00:00:00
Step 2: Processing 2023-01-24 00:00:00
Step 2: Processi